## 1. 행렬 곱셈

In [1]:
import torch

x = torch.FloatTensor([[1,2],
                        [3,4],
                        [5,6]])
y = torch.FloatTensor([[1,2],
                        [3,4]])

print(x.size(),y.size())

torch.Size([3, 2]) torch.Size([2, 2])


In [ ]:
z = torch.matmul(x,y)   # 행렬곱 연산
z.size()

torch.Size([3, 2])

In [4]:
z = x@y
z.size()

torch.Size([3, 2])

In [ ]:
x = torch.FloatTensor([[[1,2],
                        [3,4],
                        [5,6]],
                        [[7,8],
                        [9,10],         # (3,3,2)
                        [11,12]],
                        [[13,14],
                         [15,16],
                         [17,18]]])
y = torch.FloatTensor([[[1,2,2],
                        [1,2,2]],  
                        [[1,3,3],
                        [1,3,3]],       # (3,2,3)
                        [[1,4,4],
                         [1,4,4]]])

print(x.size(),y.size())


torch.Size([3, 3, 2]) torch.Size([3, 2, 3])


In [8]:
z =x@y
z.size()
z

tensor([[[  3.,   6.,   6.],
         [  7.,  14.,  14.],
         [ 11.,  22.,  22.]],

        [[ 15.,  45.,  45.],
         [ 19.,  57.,  57.],
         [ 23.,  69.,  69.]],

        [[ 27., 108., 108.],
         [ 31., 124., 124.],
         [ 35., 140., 140.]]])

In [10]:
# torch.bmm(x,y): 각 배치별로 x[i]@y[i]를 수행
z = torch.bmm(x,y)
z.size()
z

tensor([[[  3.,   6.,   6.],
         [  7.,  14.,  14.],
         [ 11.,  22.,  22.]],

        [[ 15.,  45.,  45.],
         [ 19.,  57.,  57.],
         [ 23.,  69.,  69.]],

        [[ 27., 108., 108.],
         [ 31., 124., 124.],
         [ 35., 140., 140.]]])

|구분|matmul(@)|bmm|
|---|---|---|
|입력차원|1D ~ ND|3D만 허용|
|batch 처리|자동해석 + 브로드캐스팅 가능|batch 크기 동일 필수|
|연산 범위|범용 행렬곱|배치 행렬곱|
|유연성|높음|낮음|

## 2. 신경망 내 내적 연산으로 보는 선형 변환
- W.T 전치를 이용해 입력 차원과 곱셈이 가능해지도록 맞춰준다.

In [15]:
x = torch.FloatTensor([[1, 2, 3]])          # (1,3)
W = torch.FloatTensor([[0.1, 0.2, 0.3],     
                       [0.4, 0.5, 0.6]])    # (2,3)

print(x.size(), W.size())

torch.Size([1, 3]) torch.Size([2, 3])


In [16]:
y = torch.matmul(x,W.T)   # (1,3)@(3,2)

y.size()

torch.Size([1, 2])

### 브로드 캐스팅   (Batch가 1이거나 없을때)
A : (3, 3, 2)
B : (2, 3)

A@B 연산시에  
A : (3, 3, 2)  
B : (1, 2, 3) => (3, 2, 3)  
  
=> (3, 3, 3)


### 브로드 캐스팅   (Batch가 이미 1이 아니거나 존재하는 경우)
A : (3, 3, 2)
B : (4, 2, 3)
  
=> batch 차원이 3과 4로 달라 계산되지 않음

In [18]:
x = torch.randn(2,3,2)
W = torch.randn(2,4,2)

y = x @ W.transpose(1,2)
y.size()

torch.Size([2, 3, 4])

W.transpose(1,2) : (2, 4, 2) -> (2, 2, 4) 차원끼리 치환  
(2, 3, 2) @ (2, 2, 4) => (2, 3, 4)  
  
이미지 계열 조작할때 주로 사용

## 4. 간단한 선형 연산 구현

In [19]:
import torch
import torch.nn as nn


x = torch.FloatTensor([[1.0, 2.0, 3.0]])   # (1, 3)
W = torch.FloatTensor([[0.1, 0.2, 0.3], 
                       [0.4, 0.5, 0.6]]) # (2, 3)

print(x.size(),y.size())

torch.Size([1, 3]) torch.Size([2, 3, 4])


In [20]:
# 편향 계산이 제거된 선형 모델
linear = nn.Linear(
    in_features= 3,     # 입력 벡터의 길이 (열 수)
    out_features= 2,    # 출력 벡터의 길이 (행 수)
    bias= False
)

In [21]:
with torch.no_grad():
    linear.weight.copy_(W) # 가중치 W값으로 모델 가중치 초기화 (주입)

In [23]:
y2 = linear(x)
print(y2.size(),y2)

torch.Size([1, 2]) tensor([[1.4000, 3.2000]], grad_fn=<MmBackward0>)


X = (1,3)  
W = (2,3)  
X @ W.T = (1,3)@(3,2) => (1,2)  
전치를 따로 안해줘도 전치를 알아서 진행 후 계산